# Partie 1 : Exploration des Transformers et Tâches NLP
### Hafsa | ENSA Al Hoceima - ID2 | Mini-Projet : Transformers et Systèmes RAG

Ce notebook explore la bibliothèque **HuggingFace Transformers** et implémente plusieurs tâches NLP :
- Chargement de modèles pré-entraînés
- Utilisation de tokenizers
- Classification de texte
- Analyse de sentiment
- Question Answering
- Résumé automatique

## 0. Installation des bibliothèques

In [3]:
!pip install transformers torch sentencepiece

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Utilisation des Tokenizers

Un **tokenizer** convertit le texte brut en tokens (unités numériques) compréhensibles par le modèle. Les tokens `[CLS]` et `[SEP]` sont des tokens spéciaux ajoutés automatiquement par BERT.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Artificial intelligence is transforming the world."

# Tokenization
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

# Encodage
ids = tokenizer.encode(text)
print("IDs:", ids)

# Décodage
decoded = tokenizer.decode(ids)
print("Décodé:", decoded)

c:\Users\Hafsa\Documents\projet_nlp_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokens: ['artificial', 'intelligence', 'is', 'transforming', 'the', 'world', '.']
IDs: [101, 7976, 4454, 2003, 17903, 1996, 2088, 1012, 102]
Décodé: [CLS] artificial intelligence is transforming the world. [SEP]


## 2. Classification de Texte

**Modèle :** `distilbert-base-uncased-finetuned-sst-2-english`

Cette tâche classe un texte en catégories (ici POSITIVE / NEGATIVE).

In [5]:
from transformers import pipeline

classifier = pipeline("text-classification")

texts = [
    "I love this movie!",
    "This product is terrible.",
    "The weather is okay today."
]

for text in texts:
    result = classifier(text)
    print(f"Texte: {text}")
    print(f"Résultat: {result[0]}\n")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1751.41it/s]


Texte: I love this movie!
Résultat: {'label': 'POSITIVE', 'score': 0.9998775720596313}

Texte: This product is terrible.
Résultat: {'label': 'NEGATIVE', 'score': 0.9997157454490662}

Texte: The weather is okay today.
Résultat: {'label': 'POSITIVE', 'score': 0.9997705817222595}



## 3. Analyse de Sentiment (multilingue)

**Modèle :** `nlptown/bert-base-multilingual-uncased-sentiment`

Ce modèle évalue le sentiment sur une échelle de 1 à 5 étoiles, et fonctionne en français.

In [6]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

texts = [
    "J'adore ce produit, il est fantastique!",
    "C'est vraiment décevant, je suis déçue.",
    "Le service est correct, rien d'exceptionnel."
]

for text in texts:
    result = sentiment(text)
    print(f"Texte: {text}")
    print(f"Résultat: {result[0]}\n")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1312.70it/s]


Texte: J'adore ce produit, il est fantastique!
Résultat: {'label': '5 stars', 'score': 0.848808228969574}

Texte: C'est vraiment décevant, je suis déçue.
Résultat: {'label': '2 stars', 'score': 0.5179010033607483}

Texte: Le service est correct, rien d'exceptionnel.
Résultat: {'label': '3 stars', 'score': 0.6634678840637207}



## 4. Question Answering

**Modèle :** `deepset/minilm-uncased-squad2`

Le modèle extrait la réponse à une question à partir d'un contexte donné.

In [7]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

tokenizer = AutoTokenizer.from_pretrained("deepset/minilm-uncased-squad2")
model = AutoModelForQuestionAnswering.from_pretrained("deepset/minilm-uncased-squad2")

context = """
Artificial intelligence is a field of computer science that aims to create machines 
capable of simulating human intelligence. It is used in many fields such as 
medicine, finance and education.
"""

questions = [
    "What is artificial intelligence?",
    "In which fields is AI used?"
]

for question in questions:
    inputs = tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    start = outputs.start_logits.argmax()
    end = outputs.end_logits.argmax() + 1
    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    )
    print(f"Q: {question}")
    print(f"A: {answer}\n")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1395.60it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/minilm-uncased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q: What is artificial intelligence?
A: a field of computer science

Q: In which fields is AI used?
A: medicine, finance and education



## 5. Résumé Automatique (Summarization)

**Modèle :** `sshleifer/distilbart-cnn-12-6`

Le modèle génère un résumé concis d'un texte long.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
model = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6")

text = """
Artificial intelligence (AI) is a wide-ranging branch of computer science concerned 
with building smart machines capable of performing tasks that typically require human 
intelligence. AI is being used today across different industries including healthcare, 
where it helps doctors diagnose diseases, finance, where it detects fraud, and education, 
where it personalizes learning. Machine learning, a subset of AI, allows computers to 
learn from data without being explicitly programmed.
"""

inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
with torch.no_grad():
    summary_ids = model.generate(inputs["input_ids"], max_length=60, min_length=20, num_beams=4)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Résumé:")
print(summary)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 358/358 [00:00<00:00, 7066.40it/s]


Résumé:
 AI is being used today across different industries including healthcare, finance, finance and education . Machine learning, a subset of AI, allows computers to learn from data without being explicitly programmed .


: 

## 6. Conclusion

Dans cette partie, nous avons exploré la bibliothèque HuggingFace Transformers et implémenté avec succès quatre tâches NLP fondamentales :

| Tâche | Modèle utilisé | Résultat |
|-------|---------------|----------|
| Classification de texte | distilbert-sst-2 | POSITIVE / NEGATIVE corrects |
| Analyse de sentiment | bert-multilingual-sentiment | Notes 1-5 étoiles correctes |
| Question Answering | minilm-uncased-squad2 | Réponses extraites correctement |
| Résumé automatique | distilbart-cnn-12-6 | Résumé cohérent généré |

Ces tâches démontrent la puissance et la simplicité d'utilisation des modèles pré-entraînés via HuggingFace, qui permettent d'obtenir des résultats de qualité sans entraînement coûteux.